# 🌍 Rapid Literature to Encyclopedia & Knowledge Graph Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semanticClimate/encyclopedia/blob/main/rapid_encyclopedia_pipeline.ipynb)

**semanticClimate Toolchain**

This notebook implements an automated, fast end-to-end pipeline to go from a scientific question to a structured, interactive Encyclopedia and Knowledge Graph in minutes:

1. **Step 1: Enter question & Search recent literature** to create a corpus (1–2 mins)
2. **Step 2: Extract & Interactively select key phrases** (1–10 mins interactive review)
3. **Step 3: Recursively clean corpus** (automatic text sanitation & relevance filtering)
4. **Step 4: Build Encyclopedia** using `encyclopedia` + Wikipedia / Wikidata (3–5 mins)
5. **Step 5: Build Knowledge Graph & Visualise** (interactive network with `vis-network`)



## ⚙️ Cell 0: Setup, Dependencies & Imports
Installs required packages in Google Colab (or configures local paths) and imports the core modules from `encyclopedia` and `semantic_corpus`.


In [ ]:
# Cell 0: Setup and Environment Detection
import sys
import os
import re
from pathlib import Path

# Detect environment (Google Colab vs Local)
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    print("🚀 Running in Google Colab. Installing dependencies and cloning encyclopedia...")
    !pip install -q amilib==1.0.0a9 requests beautifulsoup4 lxml pandas ipywidgets pyvis
    !rm -rf encyclopedia
    !git clone https://github.com/semanticClimate/encyclopedia.git
    sys.path.insert(0, '/content/encyclopedia')
else:
    print("💻 Running in local environment.")
    # Dynamically locate the local encyclopedia directory
    for candidate in [Path.cwd() / "encyclopedia", Path.cwd().parent / "encyclopedia"]:
        if candidate.exists() and (candidate / "encyclopedia").exists():
            sys.path.insert(0, str(candidate.resolve()))
            print(f"Added to sys.path: {candidate.resolve()}")
            break

# Import core tools from the encyclopedia repository
try:
    from encyclopedia.core.encyclopedia import AmiEncyclopedia
    from encyclopedia.utils.encyclopedia_builder import (
        create_dictionary_from_terms,
        enhance_dictionary_with_wikipedia,
        convert_dictionary_to_encyclopedia,
        add_wikipedia_descriptions_to_encyclopedia
    )
    print("✅ Successfully imported encyclopedia modules (AmiEncyclopedia, encyclopedia_builder)!")
except ImportError as e:
    print(f"⚠️ Import note: {e}. If in Colab, ensure the repo clone finished successfully.")



## 📚 Step 1: Search Recent Literature & Create Corpus (1–2 mins)
Enter a research query or topic, specify the publication years and number of papers. The script queries the **Europe PMC REST API** for recent open-access scientific publications, downloads metadata and full-text XML (or abstracts), and creates the initial corpus.


In [ ]:
# Cell 1: Enter question and search recent literature to create corpus (1-2 mins)
import requests
import json
import pandas as pd
from pathlib import Path
from IPython.display import display

# === User Parameters ===
QUERY = '("carbon capture" OR "carbon sequestration") AND "climate change"'
LIMIT_PAPERS = 6          # 5 to 10 papers for rapid 1-2 min execution
START_YEAR = 2022
END_YEAR = 2026
CORPUS_DIR = Path("corpus_raw")
CORPUS_DIR.mkdir(parents=True, exist_ok=True)

print(f"🔍 Searching Europe PMC for: '{QUERY}' (Years {START_YEAR}-{END_YEAR}, Limit {LIMIT_PAPERS})...")

base_url = "https://www.ebi.ac.uk/europepmc/webservices/rest"
search_query = f"({QUERY}) AND (FIRST_PDATE:[{START_YEAR} TO {END_YEAR}]) AND (OPEN_ACCESS:y)"
params = {
    "query": search_query,
    "format": "json",
    "pageSize": LIMIT_PAPERS,
    "resultType": "core"
}

response = requests.get(f"{base_url}/search", params=params, timeout=25)
response.raise_for_status()
data = response.json()
results = data.get("resultList", {}).get("result", [])

corpus_metadata = []

for paper in results:
    pmcid = paper.get("pmcid", "")
    pmid = paper.get("pmid", "")
    title = paper.get("title", "No Title")
    abstract = paper.get("abstractText", "")
    pub_date = paper.get("firstPublicationDate", "")
    journal = paper.get("journalTitle", "Unknown Journal")
    
    paper_id = pmcid if pmcid else (f"PMID{pmid}" if pmid else f"paper_{len(corpus_metadata)+1}")
    has_fulltext = False
    
    # Try downloading full-text XML when PMCID is available
    if pmcid:
        try:
            clean_id = pmcid.replace("PMC", "")
            xml_url = f"{base_url}/PMC{clean_id}/fullTextXML"
            xml_res = requests.get(xml_url, timeout=15)
            if xml_res.status_code == 200 and len(xml_res.content) > 1000:
                xml_path = CORPUS_DIR / f"{paper_id}.xml"
                xml_path.write_bytes(xml_res.content)
                has_fulltext = True
        except Exception:
            has_fulltext = False
            
    # Always save a clean text representation (title and abstract) for fast NLP extraction
    txt_path = CORPUS_DIR / f"{paper_id}.txt"
    txt_content = f"{title}\n\n{abstract}"
    txt_path.write_text(txt_content, encoding="utf-8")
    
    corpus_metadata.append({
        "ID": paper_id,
        "Title": title[:75] + ("..." if len(title) > 75 else ""),
        "Journal": journal[:30],
        "Year": pub_date[:4] if pub_date else "N/A",
        "FullText_XML": "Yes" if has_fulltext else "No (Abstract)",
    })

df_meta = pd.DataFrame(corpus_metadata)
print(f"\n✅ Corpus created with {len(corpus_metadata)} papers in '{CORPUS_DIR}':")
display(df_meta)



## 🏷️ Step 2: Select Key Phrases (1–10 mins Interactive)
This cell extracts the most prominent candidate phrases (n-grams and technical keyphrases) from the corpus, and displays an **interactive widget** (`ipywidgets`). You can check or uncheck terms and add custom phrases. Clicking **Confirm Selected Keyphrases** saves your choices for the encyclopedia.


In [ ]:
# Cell 2: Extract candidate keyphrases & interactively select terms (1-10 mins)
import re
import sys
from collections import Counter
from IPython.display import display, HTML

# 0. Safe ipywidgets import with auto-install and graceful fallback
try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except ImportError:
    try:
        import subprocess
        print("Installing missing ipywidgets package...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ipywidgets"])
        import ipywidgets as widgets
        HAS_WIDGETS = True
    except Exception:
        HAS_WIDGETS = False

# 1. Automatic Keyphrase Extraction from Corpus Texts
def extract_candidate_keyphrases(corpus_dir, top_n=30):
    stopwords = {
        'the', 'of', 'and', 'in', 'to', 'a', 'is', 'that', 'for', 'it', 'as', 'was', 'with',
        'be', 'by', 'on', 'not', 'he', 'i', 'this', 'are', 'or', 'an', 'they', 'from', 'at',
        'which', 'but', 'more', 'his', 'has', 'have', 'were', 'had', 'been', 'their', 'we',
        'can', 'all', 'there', 'when', 'one', 'use', 'study', 'using', 'also', 'results',
        'based', 'two', 'paper', 'research', 'analysis', 'high', 'et', 'al', 'such', 'well',
        'these', 'into', 'first', 'order', 'between', 'during', 'shown', 'both', 'only',
        'unknown', 'journal', 'date', 'title', 'abstract', 'doi', 'pmid', 'pmcid', 'author',
        'rights', 'reserved', 'copyright', 'published', 'springer', 'elsevier', 'volume', 'issue'
    }
    
    all_text = ""
    for f in corpus_dir.glob("*.txt"):
        all_text += " " + f.read_text(encoding="utf-8", errors="ignore")
        
    words = re.findall(r'[a-zA-Z]{3,}', all_text.lower())
    
    # Bigrams
    bigrams = []
    for i in range(len(words) - 1):
        w1, w2 = words[i], words[i+1]
        if w1 not in stopwords and w2 not in stopwords:
            bigrams.append(f"{w1} {w2}")
            
    # Trigrams
    trigrams = []
    for i in range(len(words) - 2):
        w1, w2, w3 = words[i], words[i+1], words[i+2]
        if w1 not in stopwords and w3 not in stopwords:
            trigrams.append(f"{w1} {w2} {w3}")
            
    # Salient single keywords
    salient_singles = [w for w in words if w not in stopwords and len(w) > 4]
    
    counter = Counter(trigrams) + Counter(bigrams) + Counter(salient_singles)
    candidates = [phrase for phrase, count in counter.most_common(top_n * 2) 
                  if count >= 2 and len(phrase) > 3][:top_n]
    return candidates

candidate_phrases = extract_candidate_keyphrases(CORPUS_DIR, top_n=24)

# 2. Interactive Selection Widget (ipywidgets) or fallback
if HAS_WIDGETS:
    checkboxes = []
    for i, phrase in enumerate(candidate_phrases):
        # Pre-check top 6 phrases by default
        cb = widgets.Checkbox(value=(i < 6), description=phrase, layout=widgets.Layout(width='auto'))
        checkboxes.append(cb)

    grid = widgets.GridBox(checkboxes, layout=widgets.Layout(grid_template_columns="repeat(3, 260px)"))
    custom_input = widgets.Text(
        placeholder="e.g. direct air capture, biochar, carbon mineralization", 
        description="Extra terms:", 
        layout=widgets.Layout(width='780px')
    )
    confirm_btn = widgets.Button(description="Confirm Selected Keyphrases", button_style="success", icon="check")
    status_label = widgets.Label(value="Select key phrases above, or add new ones, then click Confirm.")

    selected_terms = [cb.description for cb in checkboxes if cb.value]

    def on_confirm_click(b):
        global selected_terms
        selected_terms = [cb.description for cb in checkboxes if cb.value]
        if custom_input.value.strip():
            extras = [t.strip().lower() for t in custom_input.value.split(",") if t.strip()]
            selected_terms.extend(extras)
        selected_terms = list(dict.fromkeys(selected_terms)) # Deduplicate
        status_label.value = f"Selected {len(selected_terms)} terms: {', '.join(selected_terms)}"
        print(f"\n✅ Confirmed {len(selected_terms)} Keyphrases for Encyclopedia: {selected_terms}")

    confirm_btn.on_click(on_confirm_click)

    display(HTML("<h4><b>Interactive Keyphrase Selection</b></h4><p>Choose keyphrases for your encyclopedia or add custom concepts:</p>"))
    display(grid)
    display(custom_input)
    display(confirm_btn)
    display(status_label)
else:
    # Safe text-based fallback when widgets cannot be rendered
    selected_terms = candidate_phrases[:8]
    print(f"Candidate phrases extracted ({len(candidate_phrases)}):")
    for idx, cp in enumerate(candidate_phrases, 1):
        mark = "[x]" if idx <= 8 else "[ ]"
        print(f"  {mark} {idx}. {cp}")
    print(f"\n✅ Auto-selected top {len(selected_terms)} terms: {selected_terms}")
    print("(You can override selected_terms = ['term1', 'term2'] manually if needed)")



## 🧹 Step 3: Recursively Clean Corpus (Automatic)
The recursive cleaning stage traverses all documents in the corpus directory tree, stripping HTML/XML markup, bibliographies, acknowledgments, publication boilerplate, and citation artifacts. It then filters sections based on relevance to your selected keyphrases, producing clean semantic text.


In [ ]:
# Cell 3: Recursively clean corpus (automatic)
import re
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
from IPython.display import display

CORPUS_RAW = Path("corpus_raw")
CORPUS_CLEANED = Path("corpus_cleaned")
CORPUS_CLEANED.mkdir(parents=True, exist_ok=True)

def recursively_clean_text(raw_text: str, key_terms: list) -> str:
    """
    Cleans document text recursively:
    1. Removes XML/HTML tags and scripts.
    2. Cuts off bibliographies/references and acknowledgments.
    3. Cleans whitespace, citation brackets [1, 2], and URLs.
    4. Evaluates paragraph relevance against selected key phrases.
    """
    # Strip markup
    text = BeautifulSoup(raw_text, "html.parser").get_text(separator="\n")
    
    # Strip references and citations section
    text = re.split(r'\n\s*(?:References|REFERENCES|Bibliography|Literature Cited)\s*\n', text)[0]
    
    # Strip acknowledgments and author contributions
    text = re.split(r'\n\s*(?:Acknowledgments|Acknowledgements|Author Contributions)\s*\n', text)[0]
    
    # Remove citation brackets e.g. [1, 2] and URLs
    text = re.sub(r'http[s]?://\S+', '', text)
    text = re.sub(r'\[[0-9,\s–-]+\]', '', text)
    
    # Segment into paragraphs and retain meaningful, substantive content
    paragraphs = text.split("\n")
    cleaned_paragraphs = []
    
    for p in paragraphs:
        p_clean = " ".join(p.split())
        if len(p_clean) < 60:  # Skip tiny metadata lines, page numbers
            continue
        if key_terms:
            p_lower = p_clean.lower()
            if any(term.lower() in p_lower for term in key_terms) or len(p_clean) > 140:
                cleaned_paragraphs.append(p_clean)
        else:
            cleaned_paragraphs.append(p_clean)
            
    return "\n\n".join(cleaned_paragraphs)

def recursively_clean_corpus_directory(raw_dir: Path, out_dir: Path, terms: list):
    stats = []
    for file_path in raw_dir.rglob("*"):
        if file_path.is_file() and file_path.suffix in [".txt", ".xml"]:
            raw_content = file_path.read_text(encoding="utf-8", errors="ignore")
            original_chars = len(raw_content)
            
            cleaned_content = recursively_clean_text(raw_content, terms)
            cleaned_chars = len(cleaned_content)
            
            # Maintain hierarchical relative structure
            relative_path = file_path.relative_to(raw_dir)
            target_file = out_dir / relative_path.with_suffix(".cleaned.txt")
            target_file.parent.mkdir(parents=True, exist_ok=True)
            target_file.write_text(cleaned_content, encoding="utf-8")
            
            reduction = ((original_chars - cleaned_chars) / max(original_chars, 1)) * 100
            stats.append({
                "File": file_path.name,
                "Original Chars": original_chars,
                "Cleaned Chars": cleaned_chars,
                "Boilerplate Removed": f"{reduction:.1f}%"
            })
    return stats

# Run recursive cleaning on the corpus
cleaning_stats = recursively_clean_corpus_directory(CORPUS_RAW, CORPUS_CLEANED, selected_terms)
df_clean = pd.DataFrame(cleaning_stats)

print(f"\n✅ Recursively cleaned {len(cleaning_stats)} corpus files into '{CORPUS_CLEANED}':")
display(df_clean)



## 📖 Step 4: Create Encyclopedia (3–5 mins)
Now we invoke the **`encyclopedia` package** to generate the encyclopedia:
1. Creates an `AmiDictionary` from your selected terms.
2. Queries the Wikipedia / Wikidata APIs for definitions, canonical terms, and Wikidata IDs.
3. Normalizes by Wikidata ID and aggregates synonyms.
4. Adds first-sentence definitions and rich descriptions.
5. Emits the final HTML encyclopedia and previews it inline.


In [ ]:
# Cell 4: Create Encyclopedia using the encyclopedia package (3-5 mins)
from pathlib import Path
from IPython.display import display, HTML
import pandas as pd

from encyclopedia.core.encyclopedia import AmiEncyclopedia
from encyclopedia.utils.encyclopedia_builder import (
    create_dictionary_from_terms,
    enhance_dictionary_with_wikipedia,
    convert_dictionary_to_encyclopedia,
    add_wikipedia_descriptions_to_encyclopedia
)

# Use selected keyphrases (or sensible defaults if not interactively set)
terms_to_process = selected_terms if len(selected_terms) > 0 else [
    "carbon capture", "greenhouse gas", "climate change", "carbon sequestration"
]

ENCYCLOPEDIA_TITLE = "Rapid Scientific Encyclopedia"
TEMP_DIR = Path("temp_encyclopedia")
TEMP_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_HTML = Path("encyclopedia_output.html")

print(f"{'='*60}")
print(f"Building Encyclopedia for {len(terms_to_process)} terms...")
print(f"Terms: {', '.join(terms_to_process)}")
print(f"{'='*60}\n")

# 1. Create AmiDictionary from terms
print("1. Creating AmiDictionary from terms...")
dictionary = create_dictionary_from_terms(terms_to_process, ENCYCLOPEDIA_TITLE, TEMP_DIR)

# 2. Enhance with Wikipedia & Wikidata
print("2. Querying Wikipedia & Wikidata...")
dictionary = enhance_dictionary_with_wikipedia(dictionary, verbose=False)

# 3. Convert to AmiEncyclopedia
print("3. Converting dictionary to encyclopedia...")
encyclopedia = convert_dictionary_to_encyclopedia(dictionary, TEMP_DIR, title=ENCYCLOPEDIA_TITLE)

# 4. Normalize Wikidata IDs and merge synonyms
print("4. Normalizing Wikidata IDs and merging synonyms...")
encyclopedia.normalize_by_wikidata_id()
encyclopedia.merge()

# 5. Extract first-sentence definitions and descriptions
print("5. Extracting definitions and rich descriptions...")
encyclopedia, wiki_stats = add_wikipedia_descriptions_to_encyclopedia(encyclopedia, batch_size=5, verbose=False)

# 6. Save normalized HTML encyclopedia
encyclopedia.save_wiki_normalized_html(Path(OUTPUT_HTML))
print(f"\n✅ Encyclopedia generated successfully at: {OUTPUT_HTML.resolve()}")
print(f"Total entries: {len(encyclopedia.entries)}")

# Display summary table of entries
entry_data = []
for entry in encyclopedia.entries:
    entry_data.append({
        "Term": entry.get("term", ""),
        "Wikidata ID": entry.get("wikidata_id", "N/A"),
        "Wikipedia URL": entry.get("wikipedia_url", "N/A"),
        "Synonyms": ", ".join(entry.get("synonyms", [])),
    })
df_entries = pd.DataFrame(entry_data)
display(df_entries)

# Inline HTML Preview
display(HTML(f"<h4>Encyclopedia Preview:</h4><iframe src='{OUTPUT_HTML}' width='100%' height='450px' style='border:1px solid #ccc; border-radius:8px;'></iframe>"))



## 🕸️ Step 5: Build Knowledge Graph & Visualise
This step builds an interactive **Knowledge Graph** connecting your encyclopedia terms:
- **Nodes**: Encyclopedia concepts with Wikidata identifiers and definition tooltips on hover.
- **Edges**: Concept relationships and co-occurrence frequencies discovered in the cleaned corpus texts.
- **Visualization**: Interactive physics simulation with `vis-network` (drag nodes, zoom, click).


In [ ]:
# Cell 5: Build Knowledge Graph from encyclopedia and visualise
import json
import re
from pathlib import Path
from IPython.display import display, HTML
from collections import defaultdict

# 1. Extract Nodes from Encyclopedia Entries
nodes = []
terms = [entry.get("term", "").lower() for entry in encyclopedia.entries if entry.get("term")]

for i, entry in enumerate(encyclopedia.entries):
    term = entry.get("term", "")
    desc = entry.get("description_html", "")
    # Strip HTML tags for hover tooltip
    tooltip = re.sub(r'<[^>]+>', '', desc)[:180] if desc else "No description available"
    wiki_id = entry.get("wikidata_id", "")
    
    nodes.append({
        "id": term.lower(),
        "label": term.title(),
        "title": f"<b>{term.title()}</b><br>Wikidata: {wiki_id}<br>{tooltip}...",
        "value": 15 + len(entry.get("synonyms", [])) * 5,
        "group": "Wikidata Concept" if wiki_id and "Q" in wiki_id else "General Term"
    })

# 2. Extract Edges based on Corpus Co-occurrences
corpus_text = ""
for f in Path("corpus_cleaned").glob("*.txt"):
    corpus_text += " " + f.read_text(encoding="utf-8", errors="ignore").lower()

edges = []
edge_weights = defaultdict(int)

for i in range(len(terms)):
    for j in range(i + 1, len(terms)):
        t1, t2 = terms[i], terms[j]
        # Count co-occurrences in paragraphs
        paragraphs = corpus_text.split("\n\n")
        cooc = sum(1 for p in paragraphs if t1 in p and t2 in p)
        if cooc > 0:
            edge_weights[(t1, t2)] = cooc

for (t1, t2), weight in edge_weights.items():
    edges.append({
        "from": t1,
        "to": t2,
        "value": weight,
        "title": f"Co-occurrences in cleaned corpus: {weight}"
    })

# Connect sequential terms if corpus was sparse to ensure connected graph
if len(edges) < len(nodes) - 1:
    for i in range(len(terms) - 1):
        if (terms[i], terms[i+1]) not in edge_weights and (terms[i+1], terms[i]) not in edge_weights:
            edges.append({"from": terms[i], "to": terms[i+1], "value": 1, "title": "Related topic"})

# 3. Generate Standalone Interactive vis-network HTML
graph_html_content = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8">
  <script src="https://cdnjs.cloudflare.com/ajax/libs/vis-network/9.1.2/dist/vis-network.min.js"></script>
  <style>
    #mynetwork {{
      width: 100%;
      height: 520px;
      border: 1px solid #d1d5db;
      border-radius: 8px;
      background-color: #f8fafc;
    }}
  </style>
</head>
<body>
  <div id="mynetwork"></div>
  <script>
    var nodes = new vis.DataSet({json.dumps(nodes)});
    var edges = new vis.DataSet({json.dumps(edges)});
    var container = document.getElementById('mynetwork');
    var data = {{ nodes: nodes, edges: edges }};
    var options = {{
      nodes: {{
        shape: 'dot',
        font: {{ size: 15, color: '#0f172a' }},
        borderWidth: 2,
        color: {{
          border: '#2563eb',
          background: '#93c5fd',
          highlight: {{ border: '#1d4ed8', background: '#60a5fa' }}
        }}
      }},
      edges: {{
        color: {{ color: '#94a3b8', highlight: '#2563eb' }},
        smooth: {{ type: 'continuous' }}
      }},
      physics: {{
        stabilization: false,
        barnesHut: {{
          gravitationalConstant: -2800,
          springLength: 90,
          springConstant: 0.04
        }}
      }},
      interaction: {{ hover: true, tooltipDelay: 200 }}
    }};
    var network = new vis.Network(container, data, options);
  </script>
</body>
</html>
"""

GRAPH_HTML_PATH = Path("knowledge_graph.html")
GRAPH_HTML_PATH.write_text(graph_html_content, encoding="utf-8")
print(f"✅ Knowledge Graph HTML saved to {GRAPH_HTML_PATH.resolve()}")

# Render graph in notebook cell
display(HTML("<h4>Interactive Knowledge Graph (Drag nodes, zoom, hover for definitions):</h4>"))
display(HTML(graph_html_content))

